# K-Means Clustering

This notebook introduces unsupervised learning with K-Means using a synthetic dataset.

Goals:
- Generate data without class labels.
- Scale features before distance-based clustering.
- Compare candidate cluster counts with inertia and silhouette score.
- Visualize the resulting clusters.

This is a supplementary, AI-assisted practice exercise. Explore different data shapes and parameter choices to build your own understanding.

In [ ]:
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

## 1. Generate an unlabeled dataset

K-Means does not use ground-truth labels during fitting. The generated labels are retained only to inspect the synthetic data-generation process; they are not passed to the clustering algorithm.

In [ ]:
X, generated_labels = make_blobs(
    n_samples=500,
    centers=4,
    cluster_std=0.75,
    random_state=42
)

plt.scatter(X[:, 0], X[:, 1], s=25)
plt.title("Synthetic dataset (labels not used by K-Means)")
plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.tight_layout()
plt.show()

## 2. Scale the features

Scaling is important when features have different units or ranges. The synthetic data is already on similar scales, but we apply StandardScaler to demonstrate a standard workflow.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Original shape:", X.shape)
print("Scaled feature means:", X_scaled.mean(axis=0).round(3))
print("Scaled feature standard deviations:", X_scaled.std(axis=0).round(3))

## 3. Select a candidate number of clusters

Inertia measures within-cluster squared distances and generally decreases as the number of clusters increases. Silhouette score measures how well-separated the clusters are; higher values indicate better-defined clusters, though it should not be used blindly.

In [ ]:
candidate_k = range(2, 9)
inertias = []
silhouette_scores = []

for k in candidate_k:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_ids = model.fit_predict(X_scaled)

    inertias.append(model.inertia_)
    silhouette_scores.append(silhouette_score(X_scaled, cluster_ids))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(list(candidate_k), inertias, marker="o")
axes[0].set_title("Inertia (Elbow Method)")
axes[0].set_xlabel("Number of clusters (k)")
axes[0].set_ylabel("Inertia")

axes[1].plot(list(candidate_k), silhouette_scores, marker="o")
axes[1].set_title("Silhouette Score")
axes[1].set_xlabel("Number of clusters (k)")
axes[1].set_ylabel("Silhouette score")

plt.tight_layout()
plt.show()

## 4. Fit and visualize a selected model

For this synthetic dataset, we choose four clusters because that is how the data was generated. In real problems, the best value is not known in advance and should be justified using domain knowledge and multiple diagnostics.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_ids = kmeans.fit_predict(X_scaled)

print(f"Silhouette score: {silhouette_score(X_scaled, cluster_ids):.3f}")

plt.scatter(
    X_scaled[:, 0],
    X_scaled[:, 1],
    c=cluster_ids,
    cmap="viridis",
    s=25
)
centers = kmeans.cluster_centers_
plt.scatter(
    centers[:, 0],
    centers[:, 1],
    marker="X",
    s=180,
    c="red",
    edgecolor="black",
    label="Centroids"
)
plt.title("K-Means Clusters")
plt.xlabel("Scaled feature 1")
plt.ylabel("Scaled feature 2")
plt.legend()
plt.tight_layout()
plt.show()

## 5. Limitations and next steps

- K-Means assumes clusters can be represented by their means and is sensitive to outliers.
- Results can depend on initialization; `n_init` helps reduce this sensitivity.
- The silhouette score is only one diagnostic and may favor overly simple structures.
- Try changing `cluster_std`, the number of generated centers, and the random seed. Compare the resulting metrics and plots.